# <font color="#418FDE" size="6.5" uppercase>**Klassifikation vergleichen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Trainieren lineare, probabilistische, nachbarschaftsbasierte, baumbasierte und SVM-Klassifikatoren. 
- Untersuchen Wahrscheinlichkeiten, Klassen- und Stichprobengewichte sowie Voting. 
- Vergleichen Klassifikatoren mit Entscheidungsgrenzen, Konfusionsmatrizen und Fehleranalysen. 


## **1. Lineare Klassifikatoren**

### **1.1. Logistische Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_01_01.jpg?v=1784798691" width="250">



>* Lineares Modell für verständliche Klassenvorhersagen
>* Gewichte liefern Wahrscheinlichkeiten und Erklärbarkeit

>* Lernt wichtige Merkmale und ihre Gewichte
>* Trennt Klassen durch lineare Entscheidungsgrenzen

>* Regularisierung verhindert Überanpassung und stabilisiert Gewichte
>* Schnelles, interpretierbares Vergleichsmodell für Klassifikation



In [ ]:
#@title Python-Code - Logistische Regression

# Wir trainieren eine logistische Regression.
# Wahrscheinlichkeiten zeigen die Modellunsicherheit.
# Die Grafik zeigt die lineare Entscheidungsgrenze.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ein kleiner Datensatz macht die Trennung sichtbar.
features, target = make_classification(
    n_samples=220,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42,
)

# Diese Prüfung schützt vor unerwarteten Datenformen.
if features.shape != (220, 2) or target.shape != (220,):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Die Aufteilung hält beide Klassen ähnlich vertreten.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Skalierung wird nur mit Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Die logistische Regression lernt eine lineare Grenze.
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(X_train_scaled, y_train)

# Vorhersagen und Wahrscheinlichkeiten zeigen zwei Sichtweisen.
predicted = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)[:, 1]
accuracy = accuracy_score(y_test, predicted)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit: {accuracy:.2f}")
print(f"Beispiel-Wahrscheinlichkeit für Klasse 1: {probabilities[0]:.2f}")

# Ein Gitter macht die Entscheidungsgrenze sichtbar.
x_min = X_train_scaled[:, 0].min() - 0.8
x_max = X_train_scaled[:, 0].max() + 0.8
y_min = X_train_scaled[:, 1].min() - 0.8
y_max = X_train_scaled[:, 1].max() + 0.8

# Die Modellwahrscheinlichkeit wird für jeden Gitterpunkt berechnet.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 160),
    np.linspace(y_min, y_max, 160),
)
grid = np.c_[xx.ravel(), yy.ravel()]
grid_probabilities = model.predict_proba(grid)[:, 1].reshape(xx.shape)

# Die Punkte zeigen Trainingsdaten, die Linie die Grenze.
fig, ax = plt.subplots(figsize=(7, 5))
contour = ax.contourf(xx, yy, grid_probabilities, levels=20, cmap="RdBu", alpha=0.35)
ax.contour(xx, yy, grid_probabilities, levels=[0.5], colors="black", linewidths=2)
scatter = ax.scatter(
    X_train_scaled[:, 0],
    X_train_scaled[:, 1],
    c=y_train,
    cmap="RdBu",
    edgecolor="white",
)

# Beschriftungen helfen beim Lesen der Grafik.
ax.set_title("Logistische Regression: Wahrscheinlichkeit und Grenze")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
ax.legend(*scatter.legend_elements(), title="Klasse", loc="upper right")
plt.show()



### **1.2. SGD und Naive Bayes**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_01_02.jpg?v=1784798688" width="250">



>* SGD trainiert lineare Modelle schrittweise.
>* Gut für große, laufend wachsende Datensätze.

>* Skalierung und Lernrate steuern stabiles SGD-Training
>* Regularisierung und Validierung verhindern Überanpassung

>* Probabilistische Klassifikation mit Unabhängigkeitsannahme
>* Schnell bei Texten, schwach bei Abhängigkeiten



In [ ]:
#@title Python-Code - SGD und Naive Bayes

# Wir vergleichen SGD und Naive Bayes.
# Beide Modelle lernen einfache Klassifikationsregeln.
# Die Ausgabe zeigt Genauigkeit und Beispielwahrscheinlichkeiten.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung verhindert unklare Datenformen.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung bleibt durch random_state reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# SGD braucht skalierte Merkmale für stabiles lineares Lernen.
sgd_model = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="log_loss", max_iter=2000, tol=1e-3, random_state=42),
)

# GaussianNB modelliert Merkmale probabilistisch pro Klasse.
nb_model = GaussianNB()
sgd_model.fit(X_train, y_train)
nb_model.fit(X_train, y_train)

# Beide Modelle sagen Klassen für dieselben Testdaten voraus.
sgd_predictions = sgd_model.predict(X_test)
nb_predictions = nb_model.predict(X_test)
sgd_accuracy = accuracy_score(y_test, sgd_predictions)
nb_accuracy = accuracy_score(y_test, nb_predictions)

# Wahrscheinlichkeiten zeigen die Sicherheit der ersten Testentscheidung.
sgd_probability = sgd_model.predict_proba(X_test[:1])[0, 1]
nb_probability = nb_model.predict_proba(X_test[:1])[0, 1]

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"SGD Genauigkeit: {sgd_accuracy:.3f}")
print(f"Naive Bayes Genauigkeit: {nb_accuracy:.3f}")
print(f"Wahrscheinlichkeit Klasse 1, erstes Testbeispiel: SGD {sgd_probability:.3f}, NB {nb_probability:.3f}")



### **1.3. LDA und QDA**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_01_03.jpg?v=1784798687" width="250">



>* LDA nutzt gemeinsame Streuung und lineare Grenzen
>* QDA modelliert klassenweise Streuung und gekrümmte Grenzen

>* LDA schätzt stabil gemeinsame Klassenstreuung.
>* QDA braucht mehr Daten, erkennt komplexere Muster.

>* LDA: robust, linear und gut interpretierbar
>* QDA: flexibler, aber überanpassungsgefährdet



In [ ]:
#@title Python-Code - LDA und QDA

# Dieses Beispiel vergleicht LDA und QDA.
# Beide Modelle lernen unterschiedliche Entscheidungsgrenzen.
# Die Grafik zeigt lineare und gekrümmte Trennung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from matplotlib.lines import Line2D
from sklearn.datasets import make_classification
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Wir erzeugen kleine zweidimensionale Klassifikationsdaten.
features, target = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42,
)

# Eine einfache Prüfung verhindert unklare Datenfehler.
if features.shape != (300, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Wir teilen die Daten fair in Training und Test.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# LDA nutzt eine gemeinsame Streuung aller Klassen.
lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X_train, y_train)
lda_accuracy = accuracy_score(y_test, lda_model.predict(X_test))

# QDA erlaubt jeder Klasse eine eigene Streuung.
qda_model = QuadraticDiscriminantAnalysis()
qda_model.fit(X_train, y_train)
qda_accuracy = accuracy_score(y_test, qda_model.predict(X_test))

# Wir bereiten ein Raster für Entscheidungsgrenzen vor.
x_min = features[:, 0].min() - 1.0
x_max = features[:, 0].max() + 1.0
y_min = features[:, 1].min() - 1.0
y_max = features[:, 1].max() + 1.0

# Das Raster bleibt klein und gut darstellbar.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 180),
    np.linspace(y_min, y_max, 180),
)
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Die Differenz der Wahrscheinlichkeiten zeigt beide Grenzen.
lda_scores = lda_model.predict_proba(grid_points)[:, 1].reshape(xx.shape)
qda_scores = qda_model.predict_proba(grid_points)[:, 1].reshape(xx.shape)

# Kurze Ausgaben fassen die Testleistung zusammen.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"LDA Testgenauigkeit: {lda_accuracy:.2f}")
print(f"QDA Testgenauigkeit: {qda_accuracy:.2f}")

# Eine Achse zeigt Datenpunkte und beide Entscheidungsgrenzen.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=target,
    cmap="coolwarm",
    s=28,
    alpha=0.75,
)

# Die Konturlinien markieren Wahrscheinlichkeit 0,5.
lda_line = ax.contour(xx, yy, lda_scores, levels=[0.5], colors="black")
qda_line = ax.contour(xx, yy, qda_scores, levels=[0.5], colors="green")

# Beschriftungen machen den Modellunterschied sichtbar.
legend_handles = [
    Line2D([0], [0], color="black", label="LDA: lineare Grenze"),
    Line2D([0], [0], color="green", label="QDA: gekrümmte Grenze"),
]
ax.set_title("LDA und QDA auf denselben Klassifikationsdaten")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(handles=legend_handles, loc="best")
plt.show()



## **2. Bäume und Ensembles**

### **2.1. Entscheidungsbaum verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_02_01.jpg?v=1784798680" width="250">



>* Entscheidungen entstehen durch einfache Fragen
>* Vorhersagewege bleiben gut nachvollziehbar

>* Aufteilungen machen Klassen in Blättern homogener
>* Wahrscheinlichkeiten können bei kleinen Blättern täuschen

>* Gewichte betonen seltene oder wichtige Fälle
>* Ensembles stabilisieren empfindliche Entscheidungsbäume



In [ ]:
#@title Python-Code - Entscheidungsbaum verstehen

# Wir untersuchen Wahrscheinlichkeiten eines kleinen Entscheidungsbaums.
# Blattgrößen zeigen, warum Sicherheit vorsichtig interpretiert wird.
# Die Ausgabe vergleicht Vorhersagen mit und ohne Gewichte.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Ein unausgewogener Datensatz macht Klassengewichte sichtbar.
features, target = make_classification(
    n_samples=240,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    weights=[0.85, 0.15],
    class_sep=0.9,
    random_state=42,
)

# Die Aufteilung bleibt reproduzierbar und erhält beide Klassen.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Eine kurze Prüfung schützt vor unerwarteten Datenformen.
if X_train.shape[1] != 2 or len(np.unique(y_train)) != 2:
    raise ValueError("Die Beispieldaten passen nicht zur Baum-Demonstration.")

# Der erste Baum behandelt alle Trainingsbeispiele gleich.
plain_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
plain_tree.fit(X_train, y_train)

# Der zweite Baum gewichtet die seltene Klasse stärker.
weighted_tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42,
)
weighted_tree.fit(X_train, y_train)

# Wir betrachten dieselben Testfälle in beiden Bäumen.
sample_indices = np.array([0, 1, 2, 3, 4])
plain_probabilities = plain_tree.predict_proba(X_test[sample_indices])[:, 1]
weighted_probabilities = weighted_tree.predict_proba(X_test[sample_indices])[:, 1]

# Blattgrößen zeigen, wie viele Trainingsfälle die Schätzung stützen.
plain_leaves = plain_tree.apply(X_test[sample_indices])
plain_leaf_sizes = plain_tree.tree_.n_node_samples[plain_leaves]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsanteil Klasse 1: {round(float(np.mean(y_train)), 2)}")
print("Fall | wahr | P(Klasse 1) normal | P(Klasse 1) gewichtet | Blattgröße")
for row, index in enumerate(sample_indices):
    true_label = int(y_test[index])
    plain_value = round(float(plain_probabilities[row]), 2)
    weighted_value = round(float(weighted_probabilities[row]), 2)
    leaf_size = int(plain_leaf_sizes[row])
    print(f"{row + 1} | {true_label} | {plain_value} | {weighted_value} | {leaf_size}")



### **2.2. Random Forest**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_02_02.jpg?v=1784798682" width="250">



>* Viele unterschiedliche Bäume betrachten Daten zufällig.
>* Mehrheitsvoting macht Klassifikationen stabiler.

>* Baumstimmen zeigen geschätzte Sicherheit der Vorhersage
>* Wahrscheinlichkeiten helfen, Unsicherheit zu priorisieren

>* Gewichte helfen bei seltenen wichtigen Klassen
>* Voting bleibt, Bäume lernen andere Prioritäten



In [ ]:
#@title Python-Code - Random Forest

# Wir untersuchen Random-Forest-Wahrscheinlichkeiten mit Gewichtung.
# Klassenungleichgewicht macht Wahrscheinlichkeiten besonders anschaulich.
# Die Ausgabe vergleicht Stimmen und Fehlerkosten.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

# Ein kleines künstliches Datenset enthält eine seltene Klasse.
features, target = make_classification(
    n_samples=800,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    weights=[0.9, 0.1],
    class_sep=0.9,
    random_state=42,
)

# Die Aufteilung bewahrt das Klassenverhältnis in beiden Teilen.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if X_train.shape[1] != 2 or len(np.unique(y_train)) != 2:
    raise ValueError("Die Beispieldaten müssen zwei Merkmale und zwei Klassen haben.")

# Der Forest nutzt Klassengewichte gegen die seltene Klasse.
forest = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
)

# Das Modell lernt nur aus den Trainingsdaten.
forest.fit(X_train, y_train)

# Wahrscheinlichkeiten entsprechen hier Anteilen der Baumstimmen.
predicted_labels = forest.predict(X_test)
predicted_probabilities = forest.predict_proba(X_test)
minority_probabilities = predicted_probabilities[:, 1]

# Wir betrachten einen unsicheren Testfall nahe fünfzig Prozent.
uncertainty = np.abs(minority_probabilities - 0.5)
example_index = int(np.argmin(uncertainty))
example_probability = minority_probabilities[example_index]

# Die Kennzahl gewichtet beide Klassen gleich stark.
balanced_accuracy = balanced_accuracy_score(y_test, predicted_labels)
minority_share = float(np.mean(y_train == 1))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Anteil der seltenen Klasse im Training: {minority_share:.2f}")
print(f"Balanced Accuracy mit class_weight='balanced': {balanced_accuracy:.2f}")
print(f"Beispielfall: Wahrscheinlichkeit für Klasse 1 = {example_probability:.2f}")
print(f"Vorhergesagte Klasse dieses Falls: {predicted_labels[example_index]}")



### **2.3. Boosting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_02_03.jpg?v=1784798684" width="250">



>* Einfache Modelle werden schrittweise stärker
>* Neue Modelle fokussieren frühere Fehler

>* Boosting gewichtet schwierige Fälle stärker
>* Gewichte helfen, können aber Rauschen verstärken

>* Boosting nutzt gewichtete, nacheinander gelernte Stimmen
>* Wahrscheinlichkeiten und Fehler kritisch prüfen



In [ ]:
#@title Python-Code - Boosting

# Dieses Beispiel zeigt Boosting mit Stichprobengewichten.
# Falsch gewichtete Fälle beeinflussen die nächste Entscheidung stärker.
# Die Ausgabe vergleicht Fehler und vorhergesagte Wahrscheinlichkeiten.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Wir erzeugen kleine, unausgewogene Klassifikationsdaten.
features, target = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    weights=[0.85, 0.15],
    class_sep=0.9,
    random_state=42,
)

# Die Aufteilung bleibt durch Stratifikation fair vergleichbar.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Eine einfache Prüfung verhindert unklare Datenfehler.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Trainingsdaten und Zielwerte passen nicht zusammen.")

# Minderheitsfälle erhalten hier bewusst mehr Stichprobengewicht.
sample_weight = np.where(y_train == 1, 4.0, 1.0)

# AdaBoost kombiniert viele sehr kleine Entscheidungsbäume nacheinander.
base_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
model_plain = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=40,
    learning_rate=0.6,
    random_state=42,
)

# Das zweite Modell sieht dieselben Daten mit anderen Gewichten.
model_weighted = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=40,
    learning_rate=0.6,
    random_state=42,
)

# Beide Modelle werden einmal trainiert und danach verglichen.
model_plain.fit(X_train, y_train)
model_weighted.fit(X_train, y_train, sample_weight=sample_weight)

# Wir betrachten Genauigkeit und Trefferquote der seltenen Klasse.
plain_pred = model_plain.predict(X_test)
weighted_pred = model_weighted.predict(X_test)
plain_accuracy = accuracy_score(y_test, plain_pred)
weighted_accuracy = accuracy_score(y_test, weighted_pred)

# Die Minderheitsklasse ist hier die Klasse eins.
minority_mask = y_test == 1
plain_recall = np.mean(plain_pred[minority_mask] == 1)
weighted_recall = np.mean(weighted_pred[minority_mask] == 1)

# Wahrscheinlichkeiten zeigen, wie stark das Ensemble abstimmt.
plain_probability = model_plain.predict_proba(X_test)[:, 1].mean()
weighted_probability = model_weighted.predict_proba(X_test)[:, 1].mean()

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testanteil der seltenen Klasse: {np.mean(y_test):.2f}")
print(f"Ohne Gewichte: Genauigkeit {plain_accuracy:.2f}, Recall selten {plain_recall:.2f}")
print(f"Mit Gewichten: Genauigkeit {weighted_accuracy:.2f}, Recall selten {weighted_recall:.2f}")
print(f"Mittlere P(Klasse 1): ohne {plain_probability:.2f}, mit {weighted_probability:.2f}")



## **3. SVM Bewertung Vergleich**

### **3.1. SVC Modelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_03_01.jpg?v=1784798693" width="250">



>* SVC finden robuste Entscheidungsgrenzen mit Abstand
>* Support Vectors machen Modellgrenzen interpretierbar

>* Kernelwahl formt lineare oder komplexe Grenzen
>* Flexible Modelle mit Fehleranalysen prüfen

>* Grenznahe Punkte zeigen Unsicherheit und Sonderfälle.
>* Fehleranalyse vergleicht Verwechslungen und Modellgrenzen.



In [ ]:
#@title Python-Code - SVC Modelle

# Dieses Beispiel vergleicht zwei SVC-Entscheidungsgrenzen.
# Der Kernel bestimmt die Form der Grenze.
# Die Grafik zeigt Treffer und Fehler sichtbar.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import sklearn

# Wir erzeugen kleine zweidimensionale Klassifikationsdaten.
features, target = make_moons(n_samples=240, noise=0.25, random_state=42)

# Die Aufteilung bleibt durch random_state reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

# Diese Prüfung macht die erwartete Datenform explizit.
if X_train.shape[1] != 2:
    raise ValueError("Dieses Beispiel erwartet genau zwei Merkmale.")

# Skalierung wird nur mit Trainingsdaten gelernt.
linear_svc = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0))
rbf_svc = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))

# Beide Modelle lernen dieselben Trainingsdaten.
linear_svc.fit(X_train, y_train)
rbf_svc.fit(X_train, y_train)

# Wir bewerten beide Modelle auf ungesehenen Testdaten.
linear_pred = linear_svc.predict(X_test)
rbf_pred = rbf_svc.predict(X_test)
linear_acc = accuracy_score(y_test, linear_pred)
rbf_acc = accuracy_score(y_test, rbf_pred)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Linearer SVC Testgenauigkeit: {linear_acc:.2f}")
print(f"RBF-SVC Testgenauigkeit: {rbf_acc:.2f}")

# Die Konfusionsmatrix zeigt Fehlertypen des flexibleren Modells.
ConfusionMatrixDisplay.from_predictions(y_test, rbf_pred)
plt.title("RBF-SVC: Konfusionsmatrix auf Testdaten")
plt.xlabel("Vorhergesagte Klasse")
plt.ylabel("Wahre Klasse")
plt.show()



### **3.2. Gewichte und Voting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_03_02.jpg?v=1784798695" width="250">



>* Klassengewichte betonen seltene, wichtige Klassen
>* Entscheidungsgrenzen und Fehlerarten gezielt vergleichen

>* Stichprobengewichte steuern den Einfluss einzelner Datenpunkte
>* Fehleranalyse prüft fachlich vertretbare Grenzverschiebungen

>* Voting kombiniert Modelle mit unterschiedlichen Stärken
>* Ensembles trotzdem gründlich auf Fehler prüfen



In [ ]:
#@title Python-Code - Gewichte und Voting

# Wir vergleichen Gewichte und Voting praktisch.
# Klassengewichte verschieben wichtige Fehlertypen sichtbar.
# Voting kombiniert mehrere einfache Modellentscheidungen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score

# Ein unausgewogener Datensatz macht Gewichte besonders anschaulich.
features, target = make_classification(
    n_samples=600, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, weights=[0.88, 0.12], class_sep=0.9,
    random_state=42
)

# Die Aufteilung bleibt fair durch gleiche Klassenanteile.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

# Diese Prüfung schützt vor einem ungeeigneten Demonstrationsdatensatz.
if len(np.unique(y_train)) != 2 or len(np.unique(y_test)) != 2:
    raise ValueError("Die Demonstration braucht zwei Klassen in beiden Teilen.")

# Drei Modelle zeigen ungewichtet, gewichtet und kombiniert.
plain_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42))
weighted_svm = make_pipeline(
    StandardScaler(), SVC(kernel="rbf", class_weight="balanced", random_state=42)
)

# Das Voting nutzt unterschiedliche, kleine Klassifikatoren.
voting_model = VotingClassifier(
    estimators=[("svm", weighted_svm), ("logreg", LogisticRegression(max_iter=500,
    class_weight="balanced", random_state=42)), ("tree", DecisionTreeClassifier(
    max_depth=4, class_weight="balanced", random_state=42))], voting="hard"
)

# Alle Modelle werden auf denselben Trainingsdaten gelernt.
plain_svm.fit(X_train, y_train)
weighted_svm.fit(X_train, y_train)
voting_model.fit(X_train, y_train)

# Recall der seltenen Klasse zeigt übersehene wichtige Fälle.
plain_recall = recall_score(y_test, plain_svm.predict(X_test), pos_label=1)
weighted_recall = recall_score(y_test, weighted_svm.predict(X_test), pos_label=1)
voting_recall = recall_score(y_test, voting_model.predict(X_test), pos_label=1)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Recall seltene Klasse, SVM ohne Gewicht: {plain_recall:.2f}")
print(f"Recall seltene Klasse, SVM mit Gewicht: {weighted_recall:.2f}")
print(f"Recall seltene Klasse, Voting-Ensemble: {voting_recall:.2f}")

# Die Grafik zeigt, wie Voting den Merkmalsraum einteilt.
x_min = features[:, 0].min() - 0.8
x_max = features[:, 0].max() + 0.8
y_min = features[:, 1].min() - 0.8
y_max = features[:, 1].max() + 0.8

# Ein kleines Raster reicht für eine klare Entscheidungsgrenze.
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 180), np.linspace(y_min, y_max, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_prediction = voting_model.predict(grid).reshape(xx.shape)

# Eine einzige Achse hält den Vergleich fokussiert.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, grid_prediction, alpha=0.25, levels=[-0.5, 0.5, 1.5])
scatter = ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=35, edgecolor="k")

ax.set_title("Voting-Grenze nach gewichteten Einzelentscheidungen")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(*scatter.legend_elements(), title="Klasse", loc="upper right")
plt.show()



### **3.3. Klassifikatoren im Vergleich**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_B/image_03_03.jpg?v=1784798697" width="250">



>* Modelle nach Verhalten und Fehlern vergleichen
>* Kontext bestimmt die beste Modellwahl

>* Entscheidungsgrenzen zeigen Modellverhalten sichtbar
>* Plausibilität und Stabilität praktisch prüfen

>* Konfusionsmatrizen zeigen wichtige Verwechslungen.
>* Fehlerkosten bestimmen die beste Modellwahl.



In [ ]:
#@title Python-Code - Klassifikatoren im Vergleich

# Wir vergleichen Klassifikatoren anhand sichtbarer Entscheidungsgrenzen.
# Ein kleiner Datensatz macht Modellverhalten anschaulich.
# Die Grafik zeigt unterschiedliche Fehler und Grenzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Wir erzeugen zwei gekrümmte Klassen mit etwas Rauschen.
features, target = make_moons(n_samples=300, noise=0.28, random_state=42)

# Diese Prüfung schützt vor unerwarteten Datenformen.
if features.shape != (300, 2) or target.shape != (300,):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Der Testanteil bleibt getrennt für eine faire Bewertung.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

# Vier klassische Modelle zeigen verschiedene Entscheidungslogiken.
models = {
    "Logistisch": make_pipeline(StandardScaler(), LogisticRegression(random_state=42)),
    "kNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)),
    "Baum": DecisionTreeClassifier(max_depth=4, random_state=42),
    "SVM-RBF": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.5, gamma="scale")),
}

# Wir trainieren jedes Modell und speichern seine Testgenauigkeit.
trained_models = {}
scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    scores[name] = accuracy_score(y_test, model.predict(X_test))

# Die beste Testgenauigkeit bestimmt die gezeigte Entscheidungsgrenze.
best_name = max(scores, key=scores.get)
best_model = trained_models[best_name]

# Ein feines Raster macht die Modellentscheidung im Raum sichtbar.
x_min = features[:, 0].min() - 0.5
x_max = features[:, 0].max() + 0.5
y_min = features[:, 1].min() - 0.5
y_max = features[:, 1].max() + 0.5

# Das Raster bleibt klein genug für schnelle Ausführung.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 220), np.linspace(y_min, y_max, 220)
)
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_predictions = best_model.predict(grid_points).reshape(xx.shape)

# Kurze Kennzahlen ergänzen die visuelle Fehleranalyse.
print(f"scikit-learn-Version: {sklearn.__version__}")
print("Testgenauigkeit je Modell:")
for name in sorted(scores):
    print(f"{name}: {scores[name]:.2f}")

# Die Grafik zeigt Grenze, Testpunkte und Fehlklassifikationen.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, grid_predictions, alpha=0.25, cmap="coolwarm")

# Falsch klassifizierte Testpunkte werden besonders markiert.
test_predictions = best_model.predict(X_test)
wrong = test_predictions != y_test
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", edgecolor="k", label="Testpunkte")
ax.scatter(X_test[wrong, 0], X_test[wrong, 1], facecolors="none", edgecolors="yellow", s=120, linewidths=2, label="Fehler")

# Achsen und Titel machen die Darstellung lesbar.
ax.set_title(f"Entscheidungsgrenze des besten Modells: {best_name}")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(loc="best")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Klassifikation vergleichen**</font>


In this lecture, you learned to:
- Trainieren lineare, probabilistische, nachbarschaftsbasierte, baumbasierte und SVM-Klassifikatoren. 
- Untersuchen Wahrscheinlichkeiten, Klassen- und Stichprobengewichte sowie Voting. 
- Vergleichen Klassifikatoren mit Entscheidungsgrenzen, Konfusionsmatrizen und Fehleranalysen. 

In the next Module (Module 11), we will go over 'Bewertung und Tuning'